In [21]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.ensemble import RandomForestRegressor
import joblib

In [22]:
DATA_PATH=Path('../data/processed/transport_processed.csv')
df=pd.read_csv(DATA_PATH)

X = df.select_dtypes(include=['number', 'bool']).drop(columns=['actual_arrival_delay_min', 'delayed'])

y_reg = df['actual_arrival_delay_min']

In [23]:
X_train, X_test, y_train_class, y_test_class=train_test_split(X,y_reg,test_size=0.2,random_state=42)
y_train_reg = y_reg.loc[X_train.index]
y_test_reg  = y_reg.loc[X_test.index]

# Save split indices to avoid data leakage in evaluation notebook
Path('../models').mkdir(exist_ok=True)
np.save('../models/train_idx.npy', X_train.index.to_numpy())
np.save('../models/test_idx.npy',  X_test.index.to_numpy())

print(f'Train size: {len(X_train)}  |  Test size: {len(X_test)}')

Train size: 1600  |  Test size: 400


In [24]:
rfg=RandomForestRegressor(n_estimators=200, max_depth=None, random_state=42, n_jobs=-1,)
rfg.fit(X_train, y_train_reg)
y_pred_reg = rfg.predict(X_test)

mae  = mean_absolute_error(y_test_reg, y_pred_reg)
rmse = mean_squared_error(y_test_reg, y_pred_reg) ** 0.5

print(f'MAE:  {mae:.2f} min')
print(f'RMSE: {rmse:.2f} min')

MAE:  7.96 min
RMSE: 9.51 min


In [25]:
joblib.dump(rfg, '../models/rfg_model.pkl')

['../models/rfg_model.pkl']